In [ ]:
# ============================================================
# STAGE RECIPES
# ============================================================

# STAGE 01
# teacher: baseline minimal_basic_strategy
# student: FeedForwardDoubleDQN(table_realistic_default)
# bet_multipliers=(1,)
# start_state=fresh_shoe
# distillation enabled
# freeze_playing_parts=False

# STAGE 02
# student: FeedForwardDoubleDQN(table_realistic_unknown_progress)
# warm_start desde best_eval de stage_01
# teacher_checkpoint_path = best_eval de stage_01
# start_state=unknown_progress
# bet_multipliers=(1,)
# distillation enabled

# STAGE 03
# student: RecurrentDoubleDQN(table_realistic_unknown_progress)
# warm_start desde best_eval de stage_02
# teacher_checkpoint_path = best_eval de stage_02
# distillation enabled
# reset_hidden_on_round_end=False o según experimento

# STAGE 04
# abrir betting:
# bet_multipliers=(1,2,3,4)
# freeze_playing_parts=True al inicio
# use_custom_param_groups=True
# backbone_lr pequeño, play_lr pequeño, bet_lr mayor
# distillation puede seguir en playing_only=True


In [ ]:
# configuration guide - transfer learning edition
# Este notebook sirve como plantilla para correr stages de transferencia:
# baseline -> student realista -> unknown_progress -> recurrente -> betting abierto

TRANSFER_EXPERIMENT = {
    # =========================
    # CHECKPOINTS BASE
    # =========================
    "teacher_checkpoint_path": r"notebooks/training_checkpoints/baseline_playing_feedforward_v1/best_eval.pt",
    "warm_start_checkpoint_path": r"notebooks/training_checkpoints/baseline_playing_feedforward_v1/best_eval.pt",

    # =========================
    # STAGE / OBJETIVO
    # =========================
    "stage_name": "stage_01_feedforward_realistic_playing",
    "notes": "Transfer desde baseline minimal_basic_strategy hacia table_realistic_default con distillation de playing",

    # =========================
    # TRAINING MODE
    # =========================
    "use_resume": False,          # True solo para continuar exactamente la misma corrida
    "use_warm_start": True,       # True para transfer learning
    "use_teacher": True,          # True para activar distillation
    "freeze_playing_parts": False,# True cuando quieras abrir betting y preservar playing
    "use_custom_param_groups": False,  # True si quieres LR distinto para backbone/play/bet

    # =========================
    # DISTILLATION
    # =========================
    "distillation_enabled": True,
    "distillation_mode": "q_mse",     # "q_mse" | "policy_kl" | "greedy_ce"
    "distillation_weight": 0.50,      # peso inicial
    "distillation_final_weight": 0.10,# peso final al terminar decay
    "distillation_decay_steps": 50_000,
    "distillation_temperature": 1.0,  # usado sobre todo en policy_kl
    "distillation_playing_only": True,# recomendado para teacher baseline de playing

    # =========================
    # OPTIMIZER PARAM GROUPS
    # =========================
    "backbone_lr": 1e-5,
    "play_lr": 1e-5,
    "bet_lr": 3e-4,
    "default_lr": 1e-4,
    "weight_decay": 1e-5,
    "param_group_optimizer": "adamw", # "adam" | "adamw"
}


In [ ]:
from enviroment_bj import BlackjackConfig, ObservationConfig, StartStateConfig
from loss import BellmanLossConfig, LossPhaseWeightConfig
from model.agents import FeedForwardDoubleDQN, RecurrentDoubleDQN, DuelingRecurrentDoubleDQN, AgentNetworkConfig
from training import (
    ReplayBufferConfig,
    EpsilonScheduleConfig,
    DualEpsilonConfig,
    NStepConfig,
    OptimizationConfig,
    TargetUpdateConfig,
    EvaluationConfig,
    CheckpointConfig,
    PrintConfig,
    TrainerConfig,
    TrainingPipelineConfig,
    TransferLearningConfig,
    DistillationConfig,
    train_model,
    freeze_playing_policy_parts,
    build_optimizer_with_param_groups,
)

# ============================================================
# 1) OBSERVATION / TABLE
# ============================================================

observation_config = ObservationConfig(
    profile="table_realistic_default",                # minimal_basic_strategy | table_realistic_default | table_realistic_unknown_progress | fully_observable_sim
    obs_include_table_rules=True,
    obs_include_visible_rules_only=True,
    obs_include_hidden_rules=False,
    obs_include_decision_phase=True,
    obs_include_available_bet_multipliers=True,
    obs_current_hand_mode="table_raw",                # basic_strategy | table_raw
    obs_include_other_player_hands=True,
    obs_include_current_bet=True,
    obs_include_betting_context=True,
    obs_include_hand_context=True,
    obs_include_insurance_context=True,
    obs_include_temporal_context=True,
    obs_include_hands_since_shuffle=True,
    obs_include_estimated_shoe_progress=True,
    obs_include_last_hand_outcome=False,
    obs_include_recent_actions=False,
    obs_recent_actions_window=5,
    obs_include_observed_cards_history=True,
    obs_observed_cards_mode="rank_counts",            # rank_counts | low_neutral_high | recent_cards_sequence
    obs_recent_cards_window=20,
    obs_reset_history_on_shuffle=True,
    obs_include_exact_shoe_composition=False,
    obs_include_discard_summary=True,
    obs_include_n_decks=False,
    obs_include_shoe_penetration_rule=False,
)

start_state_config = StartStateConfig(
    mode="fresh_shoe",                                # fresh_shoe | unknown_progress
    min_burned_rounds=0,
    max_burned_rounds=0,
    clear_visible_histories_after_burn=True,
    hide_reshuffle_progress_from_observation=False,
)

blackjack_config = BlackjackConfig(
    n_decks=1,
    shoe_penetration=1.0,
    use_cut_card=False,
    dealer_hits_soft_17=False,
    blackjack_payout=1.5,
    dealer_peeks_for_blackjack=True,
    double_allowed_on="any_two_cards",
    double_after_split_allowed=True,
    double_split_aces_allowed=False,
    split_rule="same_value",
    max_hands_after_split=4,
    max_split_depth_per_hand=None,
    resplit_aces_allowed=True,
    hit_split_aces_allowed=False,
    surrender_allowed=True,
    insurance_allowed=True,
    six_card_charlie_enabled=False,
    base_bet=1.0,
    bet_multipliers=(1,),                             # en stage playing conviene arrancar con (1,)
    observation=observation_config,
)


In [ ]:
# ============================================================
# 2) MODEL / PIPELINE CONFIG
# ============================================================

# Stage 1 recomendado:
# teacher: baseline feedforward minimal_basic_strategy
# student: feedforward table_realistic_default
model = FeedForwardDoubleDQN.from_profile(
    "table_realistic_default",
    feedforward_hidden_dims=(256, 256),
    activation="relu",
    use_layer_norm=False,
    dropout=0.0,
    use_phase_adapters=False,
    use_module_gating=False,
)

pipeline_config = TrainingPipelineConfig(
    trainer=TrainerConfig(
        total_epochs=30,
        env_steps_per_epoch=5_000,
        train_frequency=4,
        updates_per_train_step=1,
        max_updates_per_epoch=None,
        device="auto",
        seed=13,
        reset_hidden_on_round_end=False,
        sequence_end_on_done=False,
        flush_partial_sequences_at_epoch_end=True,
        loss=BellmanLossConfig(
            gamma=0.99,
            loss_type="huber",
            phase_weights=LossPhaseWeightConfig(
                enabled=False,
                betting_weight=1.0,
                playing_weight=1.0,
            ),
        ),
    ),
    replay_buffer=ReplayBufferConfig(
        capacity=50_000,
        batch_size=64,
        warmup_size=1_000,
        sequence_length=8,
        min_sequence_length=2,
    ),
    epsilon=DualEpsilonConfig(
        betting=EpsilonScheduleConfig(
            start=1.0,
            end=0.10,
            decay_steps=40_000,
            evaluation_epsilon=0.0,
        ),
        playing=EpsilonScheduleConfig(
            start=1.0,
            end=0.03,
            decay_steps=25_000,
            evaluation_epsilon=0.0,
        ),
    ),
    n_step=NStepConfig(
        enabled=False,
        n_steps=3,
    ),
    optimization=OptimizationConfig(
        optimizer="adam",
        learning_rate=1e-3,
        weight_decay=0.0,
        scheduler="none",
        scheduler_step_size=1_000,
        scheduler_gamma=0.99,
        gradient_clipping=True,
        max_grad_norm=5.0,
    ),
    target_update=TargetUpdateConfig(
        mode="hard",
        hard_update_interval=250,
        soft_tau=0.005,
    ),
    evaluation=EvaluationConfig(
        enabled=True,
        every_n_epochs=1,
        num_rounds=1200,
        max_decisions=100_000,
    ),
    checkpoints=CheckpointConfig(
        directory=r"training_checkpoints/stage_01_feedforward_realistic_playing",
        save_latest=True,
        save_best_eval=True,
        save_periodic=True,
        periodic_interval_updates=2_500,
        best_metric_name="ev_per_1000_hands",
        maximize_best_metric=True,
    ),
    transfer=TransferLearningConfig(
        enabled=True,
        teacher_checkpoint_path=TRANSFER_EXPERIMENT["teacher_checkpoint_path"],
        warm_start_checkpoint_path=TRANSFER_EXPERIMENT["warm_start_checkpoint_path"],
        distillation=DistillationConfig(
            enabled=TRANSFER_EXPERIMENT["distillation_enabled"],
            mode=TRANSFER_EXPERIMENT["distillation_mode"],
            weight=TRANSFER_EXPERIMENT["distillation_weight"],
            final_weight=TRANSFER_EXPERIMENT["distillation_final_weight"],
            decay_steps=TRANSFER_EXPERIMENT["distillation_decay_steps"],
            temperature=TRANSFER_EXPERIMENT["distillation_temperature"],
            playing_only=TRANSFER_EXPERIMENT["distillation_playing_only"],
        ),
    ),
    prints=PrintConfig(
        enable=True,
        print_run_summary=True,
        print_warmup_interval=200,
        print_update_interval=100,
        print_collection_interval=500,
        print_epoch_header=True,
        print_epoch_summary=True,
        print_eval_summary=True,
        include_segment_details=False,
    ),
)

print(model)
print(pipeline_config)


In [ ]:
# ============================================================
# 3) OPTIONAL: FREEZE / PARAM GROUPS
# ============================================================

optimizer = None

if TRANSFER_EXPERIMENT["freeze_playing_parts"]:
    freeze_playing_policy_parts(model)

if TRANSFER_EXPERIMENT["use_custom_param_groups"]:
    optimizer = build_optimizer_with_param_groups(
        model,
        backbone_lr=TRANSFER_EXPERIMENT["backbone_lr"],
        play_lr=TRANSFER_EXPERIMENT["play_lr"],
        bet_lr=TRANSFER_EXPERIMENT["bet_lr"],
        default_lr=TRANSFER_EXPERIMENT["default_lr"],
        weight_decay=TRANSFER_EXPERIMENT["weight_decay"],
        optimizer_name=TRANSFER_EXPERIMENT["param_group_optimizer"],
    )

# NOTA:
# - freeze_playing_parts=True tiene más sentido cuando abras bet_multipliers=(1,2,3,4)
# - para el primer stage de playing normalmente lo dejaría en False


In [ ]:
# ============================================================
# 4) TRAINING CALLS
# ============================================================

# A) TRANSFER LEARNING RECOMENDADO
# Usa warm start + teacher, pero NO resume optimizer/trainer state.
result = train_model(
    envs=envs,
    model=model,
    pipeline_config=pipeline_config,
    optimizer=optimizer,    # None si no usas param groups
)

# Puedes inspeccionar qué cargó el warm start:
print(result.get("warm_start_report"))
print(result["checkpoint_dir"])


In [ ]:
# ============================================================
# 5) CUÁNDO USAR resume=True
# ============================================================
# resume=True NO es transfer learning.
# Úsalo solo para continuar exactamente la misma corrida:
#
# - mismo experimento
# - mismo tipo de entrenamiento
# - restaurar optimizer/scheduler/trainer state
#
# Ejemplo:

checkpoint_path = r"training_checkpoints/stage_01_feedforward_realistic_playing/latest.pt"

result_resume = train_model(
    envs=envs,
    model=model,
    pipeline_config=pipeline_config,
    resume=True,
    resume_checkpoint_path=checkpoint_path,
)


In [ ]:
from inference import inference_with_comparison

result = inference_with_comparison(
    checkpoint_path=MODELS_DIR / "KEEP_03C_unknown_progress_hard_feedforward_best_eval.pt",
    bet_multipliers=(1,),
    architecture="feedforward",
    feedforward_hidden_dims=(256, 256, 128),
    use_layer_norm=False,
    use_phase_adapters=False,
    use_module_gating=False,
    eval_rounds=10000,
    progress_every_n_rounds=2500,
    device="cpu",
)